In [1]:
import asyncio
import qrcode
from PIL import Image
import io
import base64
from pyzbar import pyzbar
import os
import supabase

In [2]:
from supabase import create_client, Client
from dotenv import load_dotenv

load_dotenv()

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")
assert url is not None
assert key is not None
supabase: Client = create_client(url, key)

In [3]:
from x_whatsapp import WhatsappClient

client = WhatsappClient()
await client.initialize_playwright()
await client.login()

In [8]:
await client.fetch_latest_message()

{'name': 'Prathwik',
 'recent_message': 'oh yeah',
 'time': 'Prathwik',
 'unread_messages': '0',
 'translate_y': 0.0}

In [ ]:
def callback(message):
    name = message.get("name", None)
    message = message.get("recent_message", None)
    time = message.get("time", None)
    unreads = message.get("unreads", None)

In [ ]:
import datetime

sent_messages = []


async def watch_new_messages():
    """
    Continuously watches for new messages and scheduled messages and yields them as they arrive,
    excluding messages sent by this script.
    """
    new_message = await client.fetch_latest_message()

    assert new_message is not None

    name = new_message.get("name", None)
    message = new_message.get("recent_message", None)
    time = new_message.get("time", None)

    iteration_count = 0

    while True:
        try:
            # Check for new WhatsApp messages
            new_message = await client.fetch_latest_message()
            if new_message:
                if not (
                    (
                        new_message.get("message") in sent_messages
                        or new_message.get("message") == "N/A"
                    )
                    and new_message.get("name", None) == name
                ):
                    name = new_message.get("name", None)
                    message = new_message.get("message", None)
                    time = new_message.get("time", None)
                    yield new_message

            # Check for scheduled messages every 30 iterations
            if iteration_count % 30 == 0:
                current_time = datetime.datetime.now()
                response = (
                    supabase.table("schedule")
                    .select("*")
                    .order("send_time")
                    .limit(1)
                    .execute()
                )
                if response.data:
                    scheduled_message = response.data[0]
                    scheduled_time = datetime.datetime.fromisoformat(
                        scheduled_message["send_time"]
                    )

                    if scheduled_time <= current_time:
                        yield {
                            "name": scheduled_message["recipient"],
                            "message": scheduled_message["chat"],
                            "time": scheduled_time.strftime("%I:%M %p"),
                            "scheduled": True,
                        }
                        # Remove the scheduled message from the database
                        supabase.table("schedule").delete().eq(
                            "id", scheduled_message["id"]
                        ).execute()

                        name = scheduled_message["recipient"]
                        message = scheduled_message["chat"]

            iteration_count += 1
        except Exception as e:
            print(f"Error while fetching messages: {e}")

        await asyncio.sleep(1)

In [ ]:
from gradio_client import Client

client = Client("http://127.0.0.1:7860/")

In [ ]:
async for message in watch_new_messages():
    print("New message received:")
    print(message)
    if message.get("scheduled", False):

        # Handle scheduled message
        await client.send_message(message["name"], message["message"])
    else:
        # Handle regular WhatsApp message
        result = client.predict(
            user_input=str(message["message"]),
            recipient=message["name"],
            chat_history="",
            timestamp=message["time"],
            api_name="/predict",
        )
        await client.send_message(message["name"], result)

## Closing the browser and saving the context


In [ ]:
await context.storage_state(path="browser_context.json")

if browser:
    await browser.close()
elif context:
    await context.close()

await playwright.stop()

In [ ]:
page2 = await browser.new_page()